# RAG Evaluation Experiments
## Retrieval quality measurement with IR metrics

This notebook demonstrates how to **measure retrieval quality** using a ground-truth
question set and four standard Information Retrieval metrics.

Everything runs in **mock mode** — no API keys, no model downloads, no external services.

---

### What you will see

| Step | Action | Output |
|------|--------|--------|
| 1 | Build a mock-mode index | `build_index()` result dict |
| 2 | Load ground truth | `List[GroundTruthEntry]` |
| 3 | Run `RetrievalEvaluator` | per-question result dicts |
| 4 | Per-question results | table of scores per query |
| 5 | Summary metrics | averaged Recall, Precision, MRR, nDCG |
| 6 | Save CSV | `results/evaluation_results.csv` |

---

### Prerequisites

Run from the **repository root**:
```
jupyter notebook notebooks/rag_evaluation_experiments.ipynb
```

## Ground Truth Questions

The ground truth file lives at `data/ground_truth/questions.json`. Each entry has:

| Field | Description |
|-------|-------------|
| `question` | Natural-language query to issue against the index |
| `expected_answer` | Human-written reference answer (informational only) |
| `relevant_doc_id` | The document ID that contains the correct answer |

**Why doc-level relevance?**  
Retrieval evaluation is done at the **document level** (`doc_id`), not chunk level.
A question is considered answered if at least one chunk from the relevant document
appears in the top-K results. This is the standard IR evaluation protocol.

The ground truth file supports both `relevant_doc_id` (single string) and
`relevant_doc_ids` (list) — the loader normalises both to `List[str]` internally.

In [26]:
# ---------------------------------------------------------------------------
# Environment setup — run this cell first
# ---------------------------------------------------------------------------
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'Working directory : {os.getcwd()}')
print(f'Python            : {sys.version.split()[0]}')

Working directory : c:\Users\ORENS\rag-retrieval-evaluation-lab
Python            : 3.13.9


In [27]:
# Preview the raw ground truth file
import json

GT_PATH = os.path.join('data', 'ground_truth', 'questions.json')

with open(GT_PATH, 'r', encoding='utf-8') as fh:
    raw = json.load(fh)

print(f'Ground truth file : {GT_PATH}')
print(f'Number of entries : {len(raw)}')
print()

for i, entry in enumerate(raw):
    print(f'[{i+1}] question         : {entry["question"]}')
    doc_id = entry.get('relevant_doc_ids', entry.get('relevant_doc_id', '?'))
    print(f'    relevant_doc_id  : {doc_id}')
    print()

Ground truth file : data\ground_truth\questions.json
Number of entries : 5

[1] question         : What is the recommended chunk overlap baseline?
    relevant_doc_id  : doc_001

[2] question         : What chunking strategies does the system support?
    relevant_doc_id  : doc_001

[3] question         : Which embedding providers are available?
    relevant_doc_id  : doc_001

[4] question         : What vector stores are supported?
    relevant_doc_id  : doc_001

[5] question         : How does mock mode work?
    relevant_doc_id  : doc_001



## Metrics Explained

All four metrics operate on **doc-id-based binary relevance**: a retrieved position
is a hit if its `doc_id` appears in the ground-truth `relevant_doc_ids` list.

Multi-chunk documents are **deduplicated by first occurrence** before scoring —
without this, the same document appearing as 5 chunks would inflate nDCG above 1.0.

---

### Recall @ K
```
Recall@K = (number of relevant docs in top-K) / (total number of relevant docs)
```
Answers: *"Did we find all the relevant documents?"*  
For a single-relevant-document setup, this is equivalent to **Hit@K**.

---

### Precision @ K
```
Precision@K = (number of relevant docs in top-K) / K
```
Answers: *"How many of our top-K results were relevant?"*  
Denominator is always K, not the number of returned results.

---

### MRR — Mean Reciprocal Rank
```
MRR = 1 / rank_of_first_relevant_result
```
Answers: *"How high did the first relevant result appear?"*  
Returns 0.0 if no relevant result is found.

---

### nDCG @ K — Normalised Discounted Cumulative Gain
```
DCG  = sum( rel_i / log2(i+1) )  for i in 1..K   (rel_i in {0, 1})
IDCG = DCG for the ideal ordering (all relevant docs ranked first)
nDCG = DCG / IDCG
```
Answers: *"How close is the ranking to the ideal?"*  
Relevant docs ranked higher contribute more to the score.

## Step 1 — Build a Mock-Mode Index

We index a single document (`doc_001.txt`) whose content directly answers all five
ground-truth questions. The `doc_id` is derived from the filename stem — `doc_001` —
which matches the `relevant_doc_id` values in `questions.json`.

**Pipeline configuration used:**
- Embedding: `MockEmbeddingProvider` (384-dim, deterministic, no API key)
- Chunker: `RecursiveChunker(chunk_size=300, chunk_overlap=50)`
- Vector store: `ChromaVectorStore` (or `FaissVectorStore` as fallback)

In [28]:
import tempfile
from src.pipeline.rag_pipeline import RAGPipeline
from src.embeddings.mock_embedding_provider import MockEmbeddingProvider
from src.chunking.recursive_chunker import RecursiveChunker
from src.observability.metrics import METRICS

METRICS.reset()

# Document content — must be named doc_001.txt so doc_id == 'doc_001'
DOC_001_TEXT = (
    'This system supports two chunking strategies: recursive and fixed-size.\n'
    'Recursive chunking splits text by natural boundaries (paragraphs, sentences)\n'
    'before falling back to word and character splits. Fixed chunking splits by\n'
    'a fixed character count with configurable overlap.\n\n'
    'The recommended chunk overlap baseline is 10 to 20 percent of the chunk size.\n'
    'For example, with chunk_size=500, an overlap of 50 to 100 characters is typical.\n\n'
    'Four embedding providers are available: mock, HuggingFace, OpenAI, and Gemini.\n'
    'Mock mode uses a deterministic SHA-256-seeded vector and requires no API key.\n'
    'It is the safe default for local and offline execution.\n\n'
    'The system supports two vector stores: ChromaDB and FAISS.\n'
    'ChromaDB persists to disk automatically. FAISS requires an explicit persist call.\n'
)

# Create temp directory for ChromaDB and document file
_tmpdir = tempfile.mkdtemp(prefix='rag_eval_')
_doc_path = os.path.join(_tmpdir, 'doc_001.txt')   # filename stem -> doc_id

with open(_doc_path, 'w', encoding='utf-8') as fh:
    fh.write(DOC_001_TEXT)

# Detect available vector store
try:
    import chromadb as _
    from src.vectorstores.chroma_store import ChromaVectorStore
    _store = ChromaVectorStore(
        persist_directory=os.path.join(_tmpdir, 'chroma'),
        collection_name='eval_demo',
    )
    _store_name = 'ChromaDB'
except ImportError:
    from src.vectorstores.faiss_store import FaissVectorStore
    _store = FaissVectorStore(persist_directory=os.path.join(_tmpdir, 'faiss'))
    _store_name = 'FAISS'

pipeline = RAGPipeline(
    embedding_provider=MockEmbeddingProvider(),
    vector_store=_store,
    chunker=RecursiveChunker(chunk_size=300, chunk_overlap=50),
)

index_result = pipeline.build_index(_doc_path)

print('Index build result:')
for key, val in index_result.items():
    print(f'  {key:<22}: {val}')
print(f'  {"vector_store_used":<22}: {_store_name}')

Index build result:
  doc_id                : doc_001
  source                : C:\Users\ORENS\AppData\Local\Temp\rag_eval_r6ag6546\doc_001.txt
  chunks_created        : 4
  embedding_provider    : mock
  vector_store          : chroma
  vector_store_used     : ChromaDB


## Step 2 — Load Ground Truth

`load_ground_truth()` reads the JSON file and returns `List[GroundTruthEntry]`.

Each `GroundTruthEntry` has:
- `question: str`
- `expected_answer: str`
- `relevant_doc_ids: List[str]`  (always a list, normalised from either JSON format)

In [29]:
from src.evaluation.ground_truth_loader import load_ground_truth

ground_truth = load_ground_truth(GT_PATH)

print(f'Loaded {len(ground_truth)} ground-truth entries')
print()

for i, entry in enumerate(ground_truth):
    print(f'[{i+1}] question        : {entry.question}')
    print(f'    relevant_doc_ids : {entry.relevant_doc_ids}')
    print()

Loaded 5 ground-truth entries

[1] question        : What is the recommended chunk overlap baseline?
    relevant_doc_ids : ['doc_001']

[2] question        : What chunking strategies does the system support?
    relevant_doc_ids : ['doc_001']

[3] question        : Which embedding providers are available?
    relevant_doc_ids : ['doc_001']

[4] question        : What vector stores are supported?
    relevant_doc_ids : ['doc_001']

[5] question        : How does mock mode work?
    relevant_doc_ids : ['doc_001']



## Step 3 — Run RetrievalEvaluator

`RetrievalEvaluator` accepts any object that exposes `query(text, top_k) -> List[Chunk]`
— the same interface as `RAGPipeline` and `Retriever`.

For each ground-truth entry:
1. Calls `pipeline.query(entry.question, top_k)` to get retrieved chunks
2. Extracts `doc_id` from each chunk
3. Deduplicates doc_ids by first occurrence (required for correct nDCG)
4. Computes Recall@K, Precision@K, MRR, nDCG@K against `entry.relevant_doc_ids`

Returns a list of per-question result dicts and an averaged summary dict.

In [30]:
from src.evaluation.evaluator import RetrievalEvaluator

TOP_K = 5

evaluator = RetrievalEvaluator(pipeline)
results, summary = evaluator.evaluate(ground_truth, top_k=TOP_K)

print(f'Evaluation complete')
print(f'  Questions evaluated : {summary["total_questions"]}')
print(f'  top_k               : {TOP_K}')
print(f'  Errors              : {summary["error_count"]}')

Evaluation complete
  Questions evaluated : 5
  top_k               : 5
  Errors              : 0


## Step 4 — Per-Question Results

Each result dict contains the question, relevant doc IDs, retrieved doc IDs,
and all four metric scores. We display them here as a readable table.

In [31]:
# Display with pandas if available, otherwise plain text
try:
    import pandas as pd
    _PANDAS = True
except ImportError:
    _PANDAS = False

if _PANDAS:
    df = evaluator.to_dataframe(results)
    # Select the most relevant columns for display
    display_cols = [
        'question', 'relevant_doc_ids',
        'recall_at_k', 'precision_at_k', 'mrr', 'ndcg_at_k',
        'error',
    ]
    print(df[display_cols].to_string(index=True))
else:
    # Plain-text fallback
    header = f'{"#":<3} {"recall":<8} {"prec":<8} {"mrr":<8} {"ndcg":<8}  question'
    print(header)
    print('-' * len(header))
    for i, r in enumerate(results):
        err = f' [ERR: {r["error"]}]' if r['error'] else ''
        q = r['question'][:55] + '...' if len(r['question']) > 55 else r['question']
        print(
            f'{i+1:<3} '
            f'{r["recall_at_k"]:<8.4f} '
            f'{r["precision_at_k"]:<8.4f} '
            f'{r["mrr"]:<8.4f} '
            f'{r["ndcg_at_k"]:<8.4f}  '
            f'{q}{err}'
        )

                                            question relevant_doc_ids  recall_at_k  precision_at_k  mrr  ndcg_at_k error
0    What is the recommended chunk overlap baseline?        [doc_001]          1.0             0.2  1.0        1.0  None
1  What chunking strategies does the system support?        [doc_001]          1.0             0.2  1.0        1.0  None
2           Which embedding providers are available?        [doc_001]          1.0             0.2  1.0        1.0  None
3                  What vector stores are supported?        [doc_001]          1.0             0.2  1.0        1.0  None
4                           How does mock mode work?        [doc_001]          1.0             0.2  1.0        1.0  None


In [32]:
# Show the retrieved doc_ids for each question
print('Retrieved doc_ids per question (after deduplication):')
print('-' * 60)
for i, r in enumerate(results):
    relevant = r['relevant_doc_ids']
    retrieved = r['retrieved_doc_ids']
    print(f'[{i+1}] {r["question"][:50]}')
    print(f'     relevant  : {relevant}')
    print(f'     retrieved : {retrieved[:TOP_K]}')
    hit = any(d in relevant for d in retrieved[:TOP_K])
    print(f'     hit@{TOP_K}    : {hit}')
    print()

Retrieved doc_ids per question (after deduplication):
------------------------------------------------------------
[1] What is the recommended chunk overlap baseline?
     relevant  : ['doc_001']
     retrieved : ['doc_001', 'doc_001', 'doc_001', 'doc_001']
     hit@5    : True

[2] What chunking strategies does the system support?
     relevant  : ['doc_001']
     retrieved : ['doc_001', 'doc_001', 'doc_001', 'doc_001']
     hit@5    : True

[3] Which embedding providers are available?
     relevant  : ['doc_001']
     retrieved : ['doc_001', 'doc_001', 'doc_001', 'doc_001']
     hit@5    : True

[4] What vector stores are supported?
     relevant  : ['doc_001']
     retrieved : ['doc_001', 'doc_001', 'doc_001', 'doc_001']
     hit@5    : True

[5] How does mock mode work?
     relevant  : ['doc_001']
     retrieved : ['doc_001', 'doc_001', 'doc_001', 'doc_001']
     hit@5    : True



## Step 5 — Summary Metrics

The summary dict contains averaged scores across all questions.
These are the numbers you compare across configurations (different chunkers,
embedding providers, vector stores, or `top_k` values).

In [33]:
print('Summary metrics (averaged over all questions):')
print('=' * 45)
print(f'  Questions evaluated  : {summary["total_questions"]}')
print(f'  top_k                : {TOP_K}')
print(f'  Embedding provider   : mock')
print(f'  Vector store         : {_store_name}')
print(f'  Chunker              : RecursiveChunker(size=300, overlap=50)')
print()
print(f'  avg_recall@{TOP_K}         : {summary["avg_recall_at_k"]:.4f}')
print(f'  avg_precision@{TOP_K}      : {summary["avg_precision_at_k"]:.4f}')
print(f'  avg_MRR              : {summary["avg_mrr"]:.4f}')
print(f'  avg_nDCG@{TOP_K}           : {summary["avg_ndcg_at_k"]:.4f}')
print()
print(f'  Errors               : {summary["error_count"]}')

# Sanity check: all scores should be in [0.0, 1.0]
for r in results:
    for metric in ('recall_at_k', 'precision_at_k', 'mrr', 'ndcg_at_k'):
        val = r[metric]
        assert 0.0 <= val <= 1.0, f'{metric}={val} out of [0,1] for question: {r["question"]}'
print('All per-question scores are in [0.0, 1.0] — assertion passed.')

Summary metrics (averaged over all questions):
  Questions evaluated  : 5
  top_k                : 5
  Embedding provider   : mock
  Vector store         : ChromaDB
  Chunker              : RecursiveChunker(size=300, overlap=50)

  avg_recall@5         : 1.0000
  avg_precision@5      : 0.2000
  avg_MRR              : 1.0000
  avg_nDCG@5           : 1.0000

  Errors               : 0
All per-question scores are in [0.0, 1.0] — assertion passed.


In [34]:
# Demonstrate each metric function directly on a single example
from src.evaluation.metrics import recall_at_k, precision_at_k, mrr, ndcg_at_k

print('Direct metric function examples (doc_id = "doc_001"):')
print()

relevant = ['doc_001']
retrieved_hit_rank1 = ['doc_001', 'doc_002', 'doc_003']
retrieved_hit_rank2 = ['doc_002', 'doc_001', 'doc_003']
retrieved_miss      = ['doc_002', 'doc_003', 'doc_004']

for label, retr in [
    ('hit at rank 1', retrieved_hit_rank1),
    ('hit at rank 2', retrieved_hit_rank2),
    ('miss         ', retrieved_miss),
]:
    print(f'  retrieved = {retr}  ({label})')
    print(f'    Recall@3    = {recall_at_k(relevant, retr, k=3):.4f}')
    print(f'    Precision@3 = {precision_at_k(relevant, retr, k=3):.4f}')
    print(f'    MRR         = {mrr(relevant, retr):.4f}')
    print(f'    nDCG@3      = {ndcg_at_k(relevant, retr, k=3):.4f}')
    print()

Direct metric function examples (doc_id = "doc_001"):

  retrieved = ['doc_001', 'doc_002', 'doc_003']  (hit at rank 1)
    Recall@3    = 1.0000
    Precision@3 = 0.3333
    MRR         = 1.0000
    nDCG@3      = 1.0000

  retrieved = ['doc_002', 'doc_001', 'doc_003']  (hit at rank 2)
    Recall@3    = 1.0000
    Precision@3 = 0.3333
    MRR         = 0.5000
    nDCG@3      = 0.6309

  retrieved = ['doc_002', 'doc_003', 'doc_004']  (miss         )
    Recall@3    = 0.0000
    Precision@3 = 0.0000
    MRR         = 0.0000
    nDCG@3      = 0.0000



## Step 6 — Save CSV Results

`save_results_csv()` writes all per-question results to a CSV file.
List-valued fields (`relevant_doc_ids`, `retrieved_doc_ids`, `retrieved_chunk_ids`)
are serialised as pipe-separated strings for readability in spreadsheets.

In [35]:
import os

# Save to results/ directory (relative to repo root)
CSV_PATH = os.path.join('results', 'evaluation_results.csv')

evaluator.save_results_csv(results, path=CSV_PATH)

print(f'CSV saved to : {CSV_PATH}')

# Preview the CSV
with open(CSV_PATH, 'r', encoding='utf-8') as fh:
    lines = fh.readlines()

print(f'Rows         : {len(lines) - 1} data rows  +  1 header row')
print()
print('Header:')
print(' ', lines[0].strip())
print()
print('Row 1:')
print(' ', lines[1].strip())

CSV saved to : results\evaluation_results.csv
Rows         : 5 data rows  +  1 header row

Header:
  question,expected_answer,relevant_doc_ids,retrieved_doc_ids,retrieved_chunk_ids,recall_at_k,precision_at_k,mrr,ndcg_at_k,top_k,error

Row 1:
  What is the recommended chunk overlap baseline?,10-20% of the chunk size is a common baseline for overlap.,doc_001,doc_001|doc_001|doc_001|doc_001,doc_001_chunk_0001|doc_001_chunk_0002|doc_001_chunk_0000|doc_001_chunk_0003,1.0,0.2,1.0,1.0,5,


In [36]:
# Display CSV with pandas if available
if _PANDAS:
    df_csv = pd.read_csv(CSV_PATH)
    score_cols = ['question', 'recall_at_k', 'precision_at_k', 'mrr', 'ndcg_at_k']
    print('CSV loaded into DataFrame:')
    print(df_csv[score_cols].to_string(index=False))
else:
    print('(pandas not available — raw CSV preview shown above)')

CSV loaded into DataFrame:
                                         question  recall_at_k  precision_at_k  mrr  ndcg_at_k
  What is the recommended chunk overlap baseline?          1.0             0.2  1.0        1.0
What chunking strategies does the system support?          1.0             0.2  1.0        1.0
         Which embedding providers are available?          1.0             0.2  1.0        1.0
                What vector stores are supported?          1.0             0.2  1.0        1.0
                         How does mock mode work?          1.0             0.2  1.0        1.0


## Limitations of Mock Embeddings

Mock embeddings are designed for **functional correctness testing**, not semantic retrieval.
Understanding their limitations helps interpret evaluation results.

### What mock embeddings do
- Produce a **deterministic 384-dim unit vector** seeded from the SHA-256 hash of the text
- Guarantee that the **same text always produces the same vector**
- Enable **offline evaluation** with no external dependencies

### What mock embeddings do NOT do
- Capture **semantic similarity** — "chunking strategy" and "splitting approach" produce
  completely unrelated vectors
- Produce meaningful **cosine similarity scores** — scores reflect hash proximity, not meaning
- Rank results by **relevance** — a chunk containing the answer may rank lower than one
  that does not

### Interpreting evaluation scores with mock mode

| Score | What it tells you |
|-------|-------------------|
| Recall@K = 1.0 | The pipeline is functioning correctly end-to-end |
| Recall@K = 0.0 | A bug exists in the pipeline (embedding mismatch, wrong doc_id, etc.) |
| Low MRR / nDCG | Expected — mock vectors don't rank relevant chunks first |

> **Key insight:** If `Recall@K = 1.0` with mock mode, the plumbing works.
> Switch to a real embedding provider to evaluate semantic retrieval quality.

## How to Extend with HuggingFace / OpenAI / Gemini

All four embedding providers share the same `EmbeddingProvider` interface:
```python
def embed(self, texts: List[str]) -> List[List[float]]: ...
```
Swapping providers requires changing exactly **one line** in the pipeline construction.

---

### HuggingFace (local, no API key)
```python
from src.embeddings.huggingface_provider import HuggingFaceEmbeddingProvider

provider = HuggingFaceEmbeddingProvider(model_name='all-MiniLM-L6-v2')
# Downloads ~90 MB model on first use; cached locally after that
```

---

### OpenAI (API key required)
```python
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'   # or set in .env

from src.embeddings.openai_provider import OpenAIEmbeddingProvider
provider = OpenAIEmbeddingProvider(model='text-embedding-3-small')
```

---

### Google Gemini (API key required)
```python
import os
os.environ['GEMINI_API_KEY'] = '...'   # or set in .env

from src.embeddings.gemini_provider import GeminiEmbeddingProvider
provider = GeminiEmbeddingProvider(model='models/embedding-001')
```

---

### Full experiment swap

Once you have a real provider, replace `MockEmbeddingProvider()` in Step 1 and
re-run all subsequent cells. The evaluation framework, metrics, and CSV export
are provider-agnostic.

```python
# Step 1 replacement (HuggingFace example)
from src.embeddings.huggingface_provider import HuggingFaceEmbeddingProvider

pipeline = RAGPipeline(
    embedding_provider=HuggingFaceEmbeddingProvider('all-MiniLM-L6-v2'),
    vector_store=ChromaVectorStore(...),
    chunker=RecursiveChunker(chunk_size=300, chunk_overlap=50),
)
index_result = pipeline.build_index(_doc_path)
results, summary = RetrievalEvaluator(pipeline).evaluate(ground_truth, top_k=5)
```

With semantic embeddings, you should see **meaningfully higher** Recall, MRR, and
nDCG compared to mock mode.

In [37]:
# ---------------------------------------------------------------------------
# Cleanup
# ---------------------------------------------------------------------------
import shutil

print('METRICS snapshot after evaluation run:')
print('-' * 40)
for key, val in METRICS.to_dict().items():
    print(f'  {key:<35}: {val}')

METRICS.reset()
shutil.rmtree(_tmpdir, ignore_errors=True)

print('\nTemporary files removed. METRICS reset.')
print(f'CSV results preserved at: {CSV_PATH}')

METRICS snapshot after evaluation run:
----------------------------------------
  documents_loaded_total             : 1
  chunks_created_total               : 4
  query_count_total                  : 5
  error_count_total                  : 0
  embedding_latency_seconds          : 0.0008
  retrieval_latency_seconds          : 0.003
  rerank_latency_seconds             : 0.0
  last_error                         : None
  selected_embedding_provider        : mock
  selected_vector_store              : chroma

Temporary files removed. METRICS reset.
CSV results preserved at: results\evaluation_results.csv
